In [ ]:
#carregar o dataset

from google.colab import files
uploaded = files.upload()

In [ ]:
#visualização do DataFrame

import pandas as pd
import numpy as np
#df = pd.read_csv()
df = pd.read_excel("dataset_rh_clean2.xlsx")
#df.head()
df[df["ID_Funcionario"] == 22]


In [ ]:
df.sample(10)

In [ ]:
#baixar o dataset

df.to_excel("dataset_rh_clean2.xlsx", index=False)
from google.colab import files

files.download("dataset_rh_clean2.xlsx")

In [ ]:

# ============================================================
# DATA CLEANING - DATASET DE RH
# ============================================================
#
# Correções realizadas:
# 1. Idade baseada na Data_Nascimento
# 2. Tempo_Empresa_Anos baseado nas datas de admissão/desligamento
# 3. Tempo_Empresa_Anos com 2 casas decimais
# 4. Gênero corrigido com base no primeiro nome
# 5. Verificação de datas inconsistentes
# 6. Verificação final das correções
# 7. Mantém todas as abas originais do Excel
#
# ============================================================


# ============================================================
# 1. IMPORTAR BIBLIOTECAS
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files


# ============================================================
# 2. FAZER UPLOAD DO DATASET
# ============================================================

uploaded = files.upload()

arquivo = list(uploaded.keys())[0]

print(f"Arquivo carregado: {arquivo}")


# ============================================================
# 3. VISUALIZAR AS ABAS DO EXCEL
# ============================================================

excel = pd.ExcelFile(arquivo)

print("\nAbas encontradas:")
print(excel.sheet_names)


# ============================================================
# 4. LER A ABA FUNCIONARIOS
# ============================================================

df = pd.read_excel(
    arquivo,
    sheet_name="Funcionarios"
)

print("\nQuantidade de funcionários:", len(df))

print("\nColunas do dataset:")
print(df.columns.tolist())


# ============================================================
# 5. PADRONIZAR COLUNAS DE DATA
# ============================================================

colunas_data = [
    "Data_Nascimento",
    "Data_Admissao",
    "Data_Desligamento",
    "Ultima_Promocao"
]

for coluna in colunas_data:

    if coluna in df.columns:

        df[coluna] = pd.to_datetime(
            df[coluna],
            errors="coerce"
        )


# ============================================================
# 6. DATA DE REFERÊNCIA
# ============================================================
#
# O código utiliza automaticamente a data atual.
#
# Funcionário ativo:
# Data atual - Data de admissão
#
# Funcionário desligado:
# Data de desligamento - Data de admissão
#
# ============================================================

data_referencia = pd.Timestamp.today().normalize()

print("\n========================================")
print("DATA DE REFERÊNCIA")
print("========================================")

print(
    data_referencia.strftime("%d/%m/%Y")
)


# ============================================================
# 7. CORRIGIR IDADE
# ============================================================

print("\n========================================")
print("CORREÇÃO DA IDADE")
print("========================================")


if (
    "Data_Nascimento" in df.columns
    and "Idade" in df.columns
):

    # Guardar idade original
    df["Idade_Original"] = df["Idade"]

    # Calcular idade corretamente
    idade_calculada = (
        data_referencia.year
        - df["Data_Nascimento"].dt.year
        - (
            (df["Data_Nascimento"].dt.month > data_referencia.month) |
            ((df["Data_Nascimento"].dt.month == data_referencia.month) & (df["Data_Nascimento"].dt.day > data_referencia.day))
        ).astype(int)
    )

    # Aplicar a idade calculada
    df["Idade"] = idade_calculada.astype("Int64")

    # Contar alterações
    alteracoes_idade = (
        df["Idade_Original"] != df["Idade"]
    ).sum()

    print(
        f"Idades corrigidas: {alteracoes_idade}"
    )


# ============================================================
# 8. CORRIGIR TEMPO DE EMPRESA
# ============================================================

print("\n========================================")
print("CORREÇÃO DO TEMPO DE EMPRESA")
print("========================================")


if (
    "Data_Admissao" in df.columns
    and "Data_Desligamento" in df.columns
):

    # Guardar valor original
    if "Tempo_Empresa_Anos" in df.columns:

        df["Tempo_Empresa_Anos_Original"] = (
            df["Tempo_Empresa_Anos"]
        )

    # --------------------------------------------------------
    # Para funcionários ativos:
    # usa a data atual
    #
    # Para funcionários desligados:
    # usa a data de desligamento
    # --------------------------------------------------------

    data_final = df["Data_Desligamento"].fillna(
        data_referencia
    )

    # Calcular diferença em dias
    diferenca_dias = (
        data_final - df["Data_Admissao"]
    ).dt.days

    # Converter dias para anos
    #
    # 365.25 considera anos bissextos
    #
    df["Tempo_Empresa_Anos"] = (
        diferenca_dias / 365.25
    )

    # Arredondar para duas casas decimais
    df["Tempo_Empresa_Anos"] = (
        df["Tempo_Empresa_Anos"]
        .round(2)
    )

    print(
        "Tempo de empresa recalculado com sucesso."
    )

    print(
        "Coluna mantida como numérica com 2 casas decimais."
    )


# ============================================================
# 9. CORRIGIR GÊNERO
# ============================================================
#
# A correção utiliza uma tabela explícita para os primeiros
# nomes encontrados no dataset.
#
# Isso evita depender do valor atualmente preenchido na coluna
# Genero, que contém inconsistências.
#
# ============================================================

print("\n========================================")
print("CORREÇÃO DO GÊNERO")
print("========================================")


# Dicionário baseado nos primeiros nomes existentes
# no dataset

genero_por_nome = {

    "Ana": "Feminino",
    "André": "Masculino",
    "Beatriz": "Feminino",
    "Bruno": "Masculino",
    "Caio": "Masculino",
    "Carlos": "Masculino",
    "Daniela": "Feminino",
    "Débora": "Feminino",
    "Eduardo": "Masculino",
    "Felipe": "Masculino",
    "Fernanda": "Feminino",
    "Gabriel": "Masculino",
    "Gabriela": "Feminino",
    "Helena": "Feminino",
    "Henrique": "Masculino",
    "Igor": "Masculino",
    "Isabela": "Feminino",
    "João": "Masculino",
    "Juliana": "Feminino",
    "Larissa": "Feminino",
    "Lucas": "Masculino",
    "Marcelo": "Masculino",
    "Mariana": "Feminino",
    "Natália": "Feminino",
    "Nicolas": "Masculino",
    "Patricia": "Feminino",
    "Rafael": "Masculino",
    "Sabrina": "Feminino",
    "Thiago": "Masculino",
    "Vanessa": "Feminino",
    "William": "Masculino",
    "Yasmin": "Feminino"
}


# Guardar gênero original
df["Genero_Original"] = df["Genero"]


# Extrair primeiro nome
df["Primeiro_Nome"] = (
    df["Nome"]
    .astype(str)
    .str.strip()
    .str.split()
    .str[0]
)


# Aplicar gênero correto
genero_corrigido = (
    df["Primeiro_Nome"]
    .map(genero_por_nome)
)


# Substituir somente onde existe correspondência
df["Genero"] = genero_corrigido.fillna(
    df["Genero"]
)


# Contar quantos registros foram corrigidos
alteracoes_genero = (
    df["Genero_Original"] != df["Genero"]
).sum()


print(
    f"Gêneros corrigidos: {alteracoes_genero}"
)


# Mostrar exemplos que foram corrigidos
correcoes_genero = df[
    df["Genero_Original"] != df["Genero"]
][
    [
        "Nome",
        "Genero_Original",
        "Genero"
    ]
]

print("\nExemplos de correções:")

display(
    correcoes_genero.head(10)
)


# ============================================================
# 10. VERIFICAR DATAS DE NASCIMENTO FUTURAS
# ============================================================

print("\n========================================")
print("VERIFICAÇÃO DAS DATAS")
print("========================================")


nascimentos_futuros = (
    df["Data_Nascimento"]
    > data_referencia
).sum()

print(
    f"Datas de nascimento futuras: "
    f"{nascimentos_futuros}"
)


# ============================================================
# 11. VERIFICAR ADMISSÕES FUTURAS
# ============================================================

admissoes_futuras = (
    df["Data_Admissao"]
    > data_referencia
).sum()

print(
    f"Datas de admissão futuras: "
    f"{admissoes_futuras}"
)


# ============================================================
# 12. VERIFICAR DESLIGAMENTOS FUTUROS
# ============================================================

desligamentos_futuros = (
    df["Data_Desligamento"]
    > data_referencia
).sum()

print(
    f"Datas de desligamento futuras: "
    f"{desligamentos_futuros}"
)


# ============================================================
# 13. VERIFICAR DESLIGAMENTO ANTES DA ADMISSÃO
# ============================================================

desligamento_invalido = (
    df["Data_Desligamento"].notna()
    &
    (
        df["Data_Desligamento"]
        <
        df["Data_Admissao"]
    )
).sum()


print(
    f"Desligamentos anteriores à admissão: "
    f"{desligamento_invalido}"
)


# ============================================================
# 14. VERIFICAÇÃO FINAL DA IDADE
# ============================================================

idade_calculada = (
    data_referencia.year
    - df["Data_Nascimento"].dt.year
    - (
        (df["Data_Nascimento"].dt.month > data_referencia.month) |
        ((df["Data_Nascimento"].dt.month == data_referencia.month) & (df["Data_Nascimento"].dt.day > data_referencia.day))
    ).astype(int)
).astype("Int64")


erros_idade = (
    df["Idade"] != idade_calculada
).sum()


print(
    f"\nInconsistências de idade após correção: "
    f"{erros_idade}"
)


# ============================================================
# 15. VERIFICAÇÃO FINAL DO TEMPO DE EMPRESA
# ============================================================

data_final = df["Data_Desligamento"].fillna(
    data_referencia
)

tempo_calculado = (
    (
        data_final
        - df["Data_Admissao"]
    ).dt.days
    / 365.25
).round(2)


erros_tempo = (
    df["Tempo_Empresa_Anos"]
    != tempo_calculado
).sum()


print(
    f"Inconsistências de tempo de empresa após correção: "
    f"{erros_tempo}"
)


# ============================================================
# 16. VERIFICAÇÃO FINAL DO GÊNERO
# ============================================================

genero_esperado = (
    df["Primeiro_Nome"]
    .map(genero_por_nome)
)


erros_genero = (
    genero_esperado.notna()
    &
    (df["Genero"] != genero_esperado)
).sum()


print(
    f"Inconsistências de gênero após correção: "
    f"{erros_genero}"
)


# ============================================================
# 17. MOSTRAR OS DADOS CORRIGIDOS
# ============================================================

print("\n========================================")
print("DADOS APÓS A LIMPEZA")
print("========================================")


colunas_visualizacao = [
    "ID_Funcionario",
    "Nome",
    "Genero",
    "Data_Nascimento",
    "Idade",
    "Data_Admissao",
    "Data_Desligamento",
    "Tempo_Empresa_Anos"
]


colunas_visualizacao = [
    coluna
    for coluna in colunas_visualizacao
    if coluna in df.columns
]


display(
    df[colunas_visualizacao].head(15)
)


# ============================================================
# 18. INFORMAÇÕES FINAIS
# ============================================================

print("\n========================================")
print("RESUMO FINAL")
print("========================================")


print(
    f"Total de funcionários: {len(df)}"
)


print(
    f"Funcionários ativos: "
    f"{df['Data_Desligamento'].isna().sum()}"
)


print(
    f"Funcionários desligados: "
    f"{df['Data_Desligamento'].notna().sum()}"
)


print(
    f"Idades corrigidas: "
    f"{alteracoes_idade}"
)


print(
    f"Tempos de empresa recalculados: "
    f"{len(df)}"
)


print(
    f"Gêneros corrigidos: "
    f"{alteracoes_genero}"
)


# ============================================================
# 19. REMOVER COLUNAS AUXILIARES
# ============================================================

colunas_auxiliares = [
    "Idade_Original",
    "Tempo_Empresa_Anos_Original",
    "Genero_Original",
    "Primeiro_Nome"
]


for coluna in colunas_auxiliares:

    if coluna in df.columns:

        df.drop(
            columns=coluna,
            inplace=True
        )


# ============================================================
# 20. GARANTIR QUE TEMPO_EMPRESA_ANOS É NUMÉRICO
# ============================================================

df["Tempo_Empresa_Anos"] = pd.to_numeric(
    df["Tempo_Empresa_Anos"],
    errors="coerce"
).round(2)


# ============================================================
# 21. GARANTIR QUE IDADE É INTEIRO
# ============================================================

df["Idade"] = pd.to_numeric(
    df["Idade"],
    errors="coerce"
).astype("Int64")


# ============================================================
# 22. LER TODAS AS ABAS ORIGINAIS
# ============================================================

abas = pd.read_excel(
    arquivo,
    sheet_name=None
)


# Substituir a aba Funcionarios pela versão limpa
abas["Funcionarios"] = df


# ============================================================
# 23. SALVAR DATASET FINAL
# ============================================================

arquivo_saida = (
    "dataset_rh_dashboard_corrigido.xlsx"
)


with pd.ExcelWriter(
    arquivo_saida,
    engine="openpyxl"
) as writer:

    for nome_aba, dados in abas.items():

        dados.to_excel(
            writer,
            sheet_name=nome_aba,
            index=False
        )


# ============================================================
# 24. CONFIRMAÇÃO
# ============================================================

print("\n========================================")
print("DATASET CORRIGIDO COM SUCESSO")
print("========================================")

print(
    f"Arquivo: {arquivo_saida}"
)


# ============================================================
# 25. DOWNLOAD
# ============================================================

files.download(
    arquivo_saida
)


Saving dataset_rh_dashboard.xlsx to dataset_rh_dashboard (1).xlsx
Arquivo carregado: dataset_rh_dashboard (1).xlsx

Abas encontradas:
['Funcionarios', 'Departamentos', 'Niveis']

Quantidade de funcionários: 500

Colunas do dataset:
['ID_Funcionario', 'Nome', 'Genero', 'Data_Nascimento', 'Idade', 'Estado', 'Cidade', 'Departamento', 'Cargo', 'Nivel', 'Data_Admissao', 'Data_Desligamento', 'Status', 'Salario', 'Tipo_Contrato', 'Modalidade', 'Avaliacao_Desempenho', 'Satisfacao', 'Horas_Treinamento', 'Promovido', 'Ultima_Promocao', 'Ferias_Dias', 'Faltas', 'Horas_Extras', 'Salario_Anterior', 'Motivo_Desligamento', 'Ano_Admissao', 'Ano_Desligamento', 'Tempo_Empresa_Anos']

DATA DE REFERÊNCIA
25/09/2026

CORREÇÃO DA IDADE
Idades corrigidas: 11

CORREÇÃO DO TEMPO DE EMPRESA
Tempo de empresa recalculado com sucesso.
Coluna mantida como numérica com 2 casas decimais.

CORREÇÃO DO GÊNERO
Gêneros corrigidos: 267

Exemplos de correções:


,Nome,Genero_Original,Genero
1,Beatriz Nascimento Pereira,Masculino,Feminino
2,Débora Rodrigues Oliveira,Masculino,Feminino
3,Felipe Nascimento Almeida,Feminino,Masculino
4,Thiago Gomes Carvalho,Feminino,Masculino
5,Helena Nascimento Araújo,Masculino,Feminino
6,Ana Araújo Rocha,Masculino,Feminino
9,Sabrina Rodrigues Rodrigues,Masculino,Feminino
11,Marcelo Costa Santos,Feminino,Masculino
12,André Almeida Nascimento,Feminino,Masculino
13,Yasmin Silva Lima,Masculino,Feminino



VERIFICAÇÃO DAS DATAS
Datas de nascimento futuras: 0
Datas de admissão futuras: 0
Datas de desligamento futuras: 0
Desligamentos anteriores à admissão: 0

Inconsistências de idade após correção: 0
Inconsistências de tempo de empresa após correção: 0
Inconsistências de gênero após correção: 0

DADOS APÓS A LIMPEZA


,ID_Funcionario,Nome,Genero,Data_Nascimento,Idade,Data_Admissao,Data_Desligamento,Tempo_Empresa_Anos
0,1,Bruno Nascimento Almeida,Masculino,1984-03-24,42,2018-05-11,NaT,8.38
1,2,Beatriz Nascimento Pereira,Feminino,1983-06-04,43,2022-01-10,NaT,4.71
2,3,Débora Rodrigues Oliveira,Feminino,1972-11-08,53,2020-08-11,2025-01-13,4.42
3,4,Felipe Nascimento Almeida,Masculino,1990-01-08,36,2021-07-16,NaT,5.19
4,5,Thiago Gomes Carvalho,Masculino,1993-04-05,33,2026-06-23,NaT,0.26
5,6,Helena Nascimento Araújo,Feminino,1977-05-14,49,2026-02-05,2026-06-12,0.35
6,7,Ana Araújo Rocha,Feminino,1971-02-12,55,2018-08-26,NaT,8.08
7,8,Patricia Rodrigues Lima,Feminino,1995-11-21,30,2023-01-24,NaT,3.67
8,9,Eduardo Almeida Nascimento,Masculino,2001-04-18,25,2023-04-22,NaT,3.43
9,10,Sabrina Rodrigues Rodrigues,Feminino,1998-03-14,28,2020-10-20,2021-10-23,1.01



RESUMO FINAL
Total de funcionários: 500
Funcionários ativos: 415
Funcionários desligados: 85
Idades corrigidas: 11
Tempos de empresa recalculados: 500
Gêneros corrigidos: 267

DATASET CORRIGIDO COM SUCESSO
Arquivo: dataset_rh_dashboard_corrigido.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df[df["ID_Funcionario"] == 22]

,ID_Funcionario,Nome,Genero,Data_Nascimento,Idade,Estado,Cidade,Departamento,Cargo,Nivel,...,Promovido,Ultima_Promocao,Ferias_Dias,Faltas,Horas_Extras,Salario_Anterior,Motivo_Desligamento,Ano_Admissao,Ano_Desligamento,Tempo_Empresa_Anos
47,48,Felipe Martins Fernandes,Masculino,1996-10-24,29,DF,Brasília,Recursos Humanos,Coordenador de RH,Coordenação,...,Não,NaT,14,1,9.7,8050,NaN,2025,NaN,1.39


- corrigido formatacao de tempo_empresa
- corrigido tempo_empresa nao bater com anos de admissao e ano de desligamento
- corrigido generos incorretos
- corrigido data de nascimento nao bater com idade atual